In [1]:
!pip install gspread oauth2client

In [3]:
import html
import ipywidgets as widgets
from IPython.display import HTML as IPyHTML, clear_output, display, Javascript
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import os
import json
import datetime

# ---------------------------------------------------------------------
# 1. CANDIDATES & SETTINGS
# ---------------------------------------------------------------------
CANDIDATES = [
    {"name": "Rigathi Gachagua",        "initials": "RG", "party": "DCP", "color": "#008000", "text_color": "#ffffff"}, # Green
    {"name": "Dr. William Ruto",        "initials": "WR", "party": "UDA", "color": "#FFD700", "text_color": "#111827"}, # Yellow (Dark text)
    {"name": "Edwin Sifuna",            "initials": "ES", "party": "Linda Mwananchi", "color": "#0000FF", "text_color": "#ffffff"}, # Blue
    {"name": "Dr. Fred Matiang'i",      "initials": "FM", "party": "Jubilee", "color": "#DC143C", "text_color": "#ffffff"}, # Red
    {"name": "Stephen Kalonzo Musyoka", "initials": "KM", "party": "Wiper", "color": "#0055A4", "text_color": "#ffffff"}, # Blue (slightly darker than Linda for distinction)
]

BACKGROUND_URL = "https://images.unsplash.com/photo-1451187580459-43490279c0fa?q=80&w=1920&auto=format&fit=crop"
SHEET_NAME = "Poll Results" # <--- Make sure this matches your Google Sheet exactly!

# ---------------------------------------------------------------------
# 2. GOOGLE SHEETS DATABASE CONNECTION
# ---------------------------------------------------------------------
def get_db():
    scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
    creds_json = os.environ.get("GOOGLE_CREDS_JSON")
    if creds_json:
        creds_dict = json.loads(creds_json)
        creds = ServiceAccountCredentials.from_json_keyfile_dict(creds_dict, scope)
    else:
        creds = ServiceAccountCredentials.from_json_keyfile_name("credentials.json", scope)

    client = gspread.authorize(creds)
    return client.open(SHEET_NAME).sheet1

def fetch_results():
    try:
        sheet = get_db()
        records = sheet.get_all_records()
        counts = {i: 0 for i in range(len(CANDIDATES))}
        for row in records:
            for i, c in enumerate(CANDIDATES):
                if row.get("Candidate") == c["name"]:
                    counts[i] += 1
        return counts
    except Exception as e:
        print("Database connection error:", e)
        return {i: 0 for i in range(len(CANDIDATES))}

def save_vote(candidate_name, party):
    try:
        sheet = get_db()
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        sheet.append_row([timestamp, candidate_name, party])
        return True
    except Exception as e:
        print("Failed to save vote:", e)
        return False

# ---------------------------------------------------------------------
# 3. HTML / CSS GENERATORS
# ---------------------------------------------------------------------
def card_result_html(c, votes, total):
    pct = (votes / total * 100) if total else 0.0
    return f"""
    <div style="text-align:right; font-family:'Poppins', sans-serif; min-width: 90px;">
      <div style="font-size:22px; font-weight:800; color:{c['color']}; line-height:1;">{pct:.1f}%</div>
      <div style="font-size:11px; color:#6b7280; margin:4px 0 6px;">{votes} vote{'s' if votes != 1 else ''}</div>
      <div style="height:6px; background:#eef0f4; border-radius:99px; overflow:hidden;">
        <div style="width:{pct:.1f}%; height:100%; border-radius:99px;
                    background:linear-gradient(90deg, {c['color']}, {c['color']}88);
                    transition:width .6s ease;"></div>
      </div>
    </div>"""

BASE_CSS = f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;600;700;800&display=swap');
body::before {{
    content: ""; position: fixed; top: 0; left: 0; width: 100vw; height: 100vh;
    background-image: url('{BACKGROUND_URL}'); background-size: cover; background-position: center;
    filter: blur(10px) brightness(0.6); z-index: -1;
}}
.app-wrapper {{ display: flex; justify-content: center; align-items: center; min-height: 100vh; padding: 20px; font-family: 'Poppins', sans-serif; }}
.poll-card {{ background: #fff; border-radius: 16px; padding: 12px 18px; margin-bottom: 12px; box-shadow: 0 4px 14px rgba(0,0,0,.08); transition: transform .15s ease; cursor: pointer; border: 2px solid transparent; }}
.poll-card.poll-open:hover {{ transform: translateY(-2px); }}
.poll-card.poll-dim {{ opacity: .6; cursor: not-allowed; }}
.poll-card.poll-selected {{ border-color: #7C3AED; }}
.poll-av.jupyter-button, .poll-av {{ border-radius: 50% !important; font-weight: 700 !important; border: none !important; }}
.poll-name.jupyter-button, .poll-name {{ background: transparent !important; border: none !important; color: #1f2937 !important; font-size: 16px !important; font-weight: 600 !important; text-align: left !important; padding: 0 !important; font-family: 'Poppins', sans-serif !important; }}
</style>
"""

HEADER_HTML = """
<div style="background:linear-gradient(135deg, #7C3AED 0%, #06B6D4 100%); color:#fff; border-radius:20px; padding:30px; margin-bottom:22px; box-shadow:0 12px 30px rgba(124,58,237,.4); text-align:center;">
  <div style="font-size:28px; font-weight:800; line-height:1.2;">🇰🇪 Public Opinion On Who Will Lead Kenya 2027</div>
  <div style="font-size:14px; opacity:.9; margin-top:8px;">Votes are permanently recorded. One vote per browser.</div>
</div>
"""

# ---------------------------------------------------------------------
# 4. INTERACTIVE LOGIC
# ---------------------------------------------------------------------
state = {"voted": False}
result_boxes, cards, click_targets = [], [], []
live_counts = fetch_results()

out = widgets.Output()

def refresh_ui():
    total = sum(live_counts.values())
    for i, box in enumerate(result_boxes):
        box.value = card_result_html(CANDIDATES[i], live_counts[i], total)

def cast_vote(i):
    if state["voted"]: return

    c = CANDIDATES[i]
    if save_vote(c["name"], c["party"]):
        state["voted"] = True
        live_counts[i] += 1

        for b in click_targets: b.disabled = True
        for j, card in enumerate(cards):
            card.remove_class("poll-open")
            card.add_class("poll-selected" if j == i else "poll-dim")

        status.description = "✅ Vote Submitted & Recorded"
        status.button_style = "success"
        refresh_ui()
        display(Javascript("localStorage.setItem('kenya_poll_voted', 'true');"))
    else:
        status.description = "⚠️ Error saving vote. Try again."

def check_if_already_voted():
    display(Javascript("""
        if (localStorage.getItem('kenya_poll_voted') === 'true') {
            document.querySelectorAll('.poll-card').forEach(c => {
                c.classList.remove('poll-open');
                c.classList.add('poll-dim');
            });
            var btn = document.getElementById('vote-status-btn');
            if(btn) { btn.innerText = "You have already voted in this poll."; }
        }
    """))

def make_card(i, c):
    avatar = widgets.Button(description=c["initials"], layout=widgets.Layout(width="50px", height="50px", margin="0 14px 0 0"))
    avatar.style.button_color = c["color"]
    avatar.style.text_color = c.get("text_color", "#ffffff") # Set text color for yellow button
    avatar.add_class("poll-av")

    name_btn = widgets.Button(description=c["name"], layout=widgets.Layout(width="100%", height="30px"))
    name_btn.add_class("poll-name")

    party_chip = widgets.HTML(f'<div style="display:flex; align-items:center; gap:6px; color:#6b7280; font-size:12px;"><span style="width:12px; height:12px; border-radius:3px; background:{c["color"]};"></span>{c["party"]}</div>')

    for b in (avatar, name_btn):
        b.on_click(lambda _b, i=i: cast_vote(i))
        click_targets.append(b)

    result = widgets.HTML("", layout=widgets.Layout(width="120px"))
    result_boxes.append(result)
    middle = widgets.VBox([name_btn, party_chip], layout=widgets.Layout(flex="1 1 auto"))
    card = widgets.HBox([avatar, middle, result], layout=widgets.Layout(width="100%", align_items="center"))
    card.add_class("poll-card"); card.add_class("poll-open")
    cards.append(card)
    return card

card_widgets = [make_card(i, c) for i, c in enumerate(CANDIDATES)]
status = widgets.Button(description="Tap a candidate to vote", disabled=True, layout=widgets.Layout(width="100%", margin="10px 0 0 0"))
status.add_class("vote-status-btn")

app_content = widgets.VBox([widgets.HTML(HEADER_HTML), *card_widgets, status], layout=widgets.Layout(max_width="600px", width="100%"))
ui = widgets.VBox([widgets.HTML(BASE_CSS), app_content], layout=widgets.Layout(width="100%", align_items="center"))
ui.add_class("app-wrapper")

# Initial render
refresh_ui()
display(ui)
check_if_already_voted()

Database connection error: [Errno 2] No such file or directory: 'credentials.json'


<IPython.core.display.Javascript object>

Failed to save vote: [Errno 2] No such file or directory: 'credentials.json'
